RAG pipelines - Data ingestion to vector DB pipeline

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"   Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"   Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

In [ ]:
all_pdf_documents 

In [ ]:
###Text splitting to get chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap= chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

Embedding and VectorStore DB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Vector Store DB

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
            "hnsw:space": "cosine"
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

In [ ]:
chunks 

In [ ]:
#Convert the text to embeddings 

texts = [doc.page_content for doc in chunks]
texts

In [ ]:
#Generate the embeddings

embeddings = embedding_manager.generate_embeddings(texts)

#Store into the vector db

vectorstore.add_documents(chunks,embeddings)
print("Documents stored:", vectorstore.collection.count())

In [ ]:
print(chunks[0].page_content[:1000])

In [ ]:
print(vectorstore.collection.peek())

In [ ]:
query_embedding = embedding_manager.generate_embeddings(
    ["Attention Is All You Need"]
)[0]

results = vectorstore.collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

print(results)

Retriever pipeline from vector store

In [ ]:
# class RAGRetriever:
#     """Handles query-based retrieval from the vector store"""
    
#     def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
#         """
#         Initialize the retriever
        
#         Args:
#             vector_store: Vector store containing document embeddings
#             embedding_manager: Manager for generating query embeddings
#         """
#         self.vector_store = vector_store
#         self.embedding_manager = embedding_manager

#     def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
#         """
#         Retrieve relevant documents for a query
        
#         Args:
#             query: The search query
#             top_k: Number of top results to return
#             score_threshold: Minimum similarity score threshold
            
#         Returns:
#             List of dictionaries containing retrieved documents and metadata
#         """
#         print(f"Retrieving documents for query: '{query}'")
#         print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
#         # Generate query embedding
#         query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
#         # Search in vector store
#         try:
#             results = self.vector_store.collection.query(
#                 query_embeddings=[query_embedding.tolist()],
#                 n_results=top_k
#             )
            
#             # Process results
#             retrieved_docs = []
            
#             if results['documents'] and results['documents'][0]:
#                 documents = results['documents'][0]
#                 metadatas = results['metadatas'][0]
#                 distances = results['distances'][0]
#                 ids = results['ids'][0]
                
#                 for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
#                     # Convert distance to similarity score (ChromaDB uses cosine distance)
#                     similarity_score = 1 - distance
                    
#                     if similarity_score >= score_threshold:
#                         retrieved_docs.append({
#                             'id': doc_id,
#                             'content': document,
#                             'metadata': metadata,
#                             'similarity_score': similarity_score,
#                             'distance': distance,
#                             'rank': i + 1
#                         })
                
#                 print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
#             else:
#                 print("No documents found")
            
#             return retrieved_docs
            
#         except Exception as e:
#             print(f"Error during retrieval: {e}")
#             return []

# rag_retriever=RAGRetriever(vectorstore,embedding_manager)

from typing import List, Dict, Any

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store,
        embedding_manager
    ):
        """
        Initialize the retriever

        Args:
            vector_store: ChromaDB vector store
            embedding_manager: Embedding generator
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: User query
            top_k: Number of documents to retrieve

        Returns:
            List of retrieved documents
        """

        print(f"\nRetrieving documents for query: '{query}'")
        print(f"Top K: {top_k}")

        try:
            # Generate query embedding
            query_embedding = (
                self.embedding_manager
                .generate_embeddings([query])[0]
            )

            # Query ChromaDB
            results = self.vector_store.collection.query(
                query_embeddings=[
                    query_embedding.tolist()
                ],
                n_results=top_k
            )

            retrieved_docs = []

            if (
                results["documents"]
                and len(results["documents"]) > 0
                and len(results["documents"][0]) > 0
            ):

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                print(
                    f"\nFound {len(documents)} matching documents\n"
                )

                for rank, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    ),
                    start=1
                ):

                    print(
                        f"Rank {rank} | Distance: {distance:.4f}"
                    )

                    retrieved_docs.append({
                        "rank": rank,
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "distance": float(distance)
                    })

            else:
                print("No documents found.")

            return retrieved_docs

        except Exception as e:
            print(
                f"Error during retrieval: {e}"
            )
            return []

In [ ]:
rag_retriever = RAGRetriever(
    vectorstore,
    embedding_manager
)

In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("what is attention is all you need")

In [ ]:
rag_retriever.retrieve('what is self-attention mechanism')

RAG pipeline - VectorDB to LLM output generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.3-70b-versatile", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [ ]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

In [ ]:
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile"
)

response = llm.invoke("What is machine learning?")
print(response.content)

In [ ]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("What is self-attention mechanism")

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke([
    HumanMessage(
        content=prompt.format(
            context=context,
            query=query
            )
        )
    ])
    return response.content

In [ ]:
answer=rag_simple("What is self-attention mechanism?",rag_retriever,llm)
print(answer)


Enchanced RAG pipeline features

In [ ]:
# # --- Enhanced RAG Pipeline Features ---
# def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
#     """
#     RAG pipeline with extra features:
#     - Returns answer, sources, confidence score, and optionally full context.
#     """
#     results = retriever.retrieve(query, top_k=top_k)
#     if not results:
#         return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
#     # Prepare context and sources
#     context = "\n\n".join([doc['content'] for doc in results])
#     sources = [{
#         'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
#         'page': doc['metadata'].get('page', 'unknown'),
#         'score': doc['similarity_score'],
#         'preview': doc['content'][:300] + '...'
#     } for doc in results]
#     confidence = max([doc['similarity_score'] for doc in results])
    
#     # Generate answer
#     prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
#     response = llm.invoke([prompt.format(context=context, query=query)])
    
#     output = {
#         'answer': response.content,
#         'sources': sources,
#         'confidence': confidence
#     }
#     if return_context:
#         output['context'] = context
#     return output

# # Example usage:
# result = rag_advanced("who is abhinav", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
# print("Answer:", result['answer'])
# print("Sources:", result['sources'])
# print("Confidence:", result['confidence'])
# print("Context Preview:", result['context'][:300])

from langchain_core.messages import HumanMessage

def rag_advanced(
    query,
    retriever,
    llm,
    top_k=7,
    return_context=False
):
    """
    Advanced RAG Pipeline

    Returns:
    - answer
    - sources
    - confidence
    - retrieved context (optional)
    """

    # Retrieve documents
    results = retriever.retrieve(
        query=query,
        top_k=top_k
    )

    if not results:
        return {
            "answer": "No relevant documents found.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }

    # Build Context
    context = "\n\n".join(
        [doc["content"] for doc in results]
    )

    # Build Source Information
    sources = []

    for doc in results:

        distance = float(doc["distance"])

        # Convert distance to similarity score
        similarity_score = max(
            0,
            round(1 - distance, 4)
        )

        sources.append({
            "source_file":
                doc["metadata"].get(
                    "source_file",
                    "unknown"
                ),

            "page":
                doc["metadata"].get(
                    "page",
                    "unknown"
                ),

            "distance":
                round(distance, 4),

            "similarity_score":
                similarity_score,

            "preview":
                doc["content"][:200]
        })

    # Confidence
    best_distance = min(
        [doc["distance"] for doc in results]
    )

    confidence = round(
        max(0, 1 - best_distance),
        4
    )

    # Prompt
    prompt = f"""
You are an expert AI researcher.

Use the provided context to answer the question.

Instructions:
1. Explain the answer in simple language.
2. Do not copy sentences directly unless necessary.
3. Provide intuition behind the concept.
4. If applicable, explain why it is important.
5. Keep the answer between 100 and 200 words.
6. Use only information from the context.

Context:
{context}

Question:
{query}

Answer:
"""

    try:

        response = llm.invoke([
            HumanMessage(content=prompt)
        ])

        answer = response.content

    except Exception as e:

        answer = f"LLM Error: {str(e)}"

    output = {
        "answer": answer,
        "sources": sources,
        "confidence": confidence
    }

    if return_context:
        output["context"] = context

    return output

In [ ]:
result = rag_advanced(
    query="What is self-attention?",
    retriever=rag_retriever,
    llm=llm,
    top_k=3,
    return_context=True
)

In [ ]:
print("\nANSWER:")
print(result["answer"])

print("\nCONFIDENCE:")
print(result["confidence"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

print("\nCONTEXT:")
print(result["context"][:1000])

In [ ]:
from typing import Dict, Any
from langchain_core.messages import HumanMessage


class AdvancedRAGPipeline:

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(
        self,
        question: str,
        top_k: int = 5,
        stream: bool = False,
        summarize: bool = False
    ) -> Dict[str, Any]:

        # Retrieve Documents
        results = self.retriever.retrieve(
            query=question,
            top_k=top_k
        )

        if not results:
            return {
                "question": question,
                "answer": "No relevant context found.",
                "sources": [],
                "summary": None,
                "confidence": 0.0,
                "history": self.history
            }

        # Build Context
        context = "\n\n".join(
            [doc["content"] for doc in results]
        )

        # Build Sources
        sources = []

        for doc in results:

            distance = float(doc["distance"])

            similarity_score = round(
                max(0, 1 - distance),
                4
            )

            sources.append({
                "source": doc["metadata"].get(
                    "source_file",
                    "unknown"
                ),

                "page": doc["metadata"].get(
                    "page",
                    "unknown"
                ),

                "distance": round(
                    distance,
                    4
                ),

                "similarity_score": similarity_score,

                "preview":
                    doc["content"][:150] + "..."
            })

        # Confidence Score
        confidence = round(
            sum(
                source["similarity_score"]
                for source in sources
            ) / len(sources),
            4
        )

        # Better Prompt
        prompt = f"""
You are an expert AI researcher.

Use the provided context to answer the question.

Instructions:
1. Explain the answer in simple language.
2. Do not copy sentences directly unless necessary.
3. Provide intuition behind the concept.
4. If applicable, explain why it is important.
5. Keep the answer between 100 and 200 words.
6. Use only information from the context.

Context:
{context}

Question:
{question}

Answer:
"""

        # Simple status message
        if stream:
            print("Retrieval complete. Generating response...")

        # Generate Answer
        response = self.llm.invoke(
            [
                HumanMessage(
                    content=prompt
                )
            ]
        )

        answer = response.content

        # Add Citations
        citations = "\n\nSources:\n"

        for i, source in enumerate(
            sources,
            start=1
        ):
            citations += (
                f"[{i}] "
                f"{source['source']} "
                f"(Page {source['page']})\n"
            )

        answer_with_citations = (
            answer +
            citations
        )

        # Summarization
        summary = None

        if summarize:

            summary_prompt = f"""
Summarize the following answer in 2 concise sentences.

Answer:
{answer}
"""

            summary_response = self.llm.invoke(
                [
                    HumanMessage(
                        content=summary_prompt
                    )
                ]
            )

            summary = summary_response.content

        # Save History
        self.history.append({
            "question": question,
            "answer": answer,
            "summary": summary,
            "confidence": confidence,
            "sources": sources
        })

        # Return Output
        return {
            "question": question,
            "answer": answer_with_citations,
            "summary": summary,
            "confidence": confidence,
            "sources": sources,
            "history": self.history
        }

In [ ]:
adv_rag = AdvancedRAGPipeline(
    rag_retriever,
    llm
)

result = adv_rag.query(
    "What is self-attention?",
    top_k=5,
    stream=True,
    summarize=True
)

print("\nANSWER:\n")
print(result["answer"])

print("\nCONFIDENCE:")
print(result["confidence"])

print("\nSUMMARY:\n")
print(result["summary"])